# Speech Transcription, Word Alignment & Transcript Re-alignment

One facade — `SpeechAlignmentPipeline` — hides the whisperX / forced-alignment /
fuzzy-matching machinery and returns **structured dataclasses** you can act on
(e.g. drive audio cutting in another project).

| Task | Method | Structured output |
|---|---|---|
| 1. Audio → transcript | `SpeechAlignmentPipeline.transcribe` | `str` |
| 2. Audio → word timestamps | `SpeechAlignmentPipeline.transcribe_words` | `list[Word]` |
| 3. Check an annotation | `TranscriptEvaluator.evaluate` (or `pipe.evaluate_annotation`) | `TranscriptMetrics` + `is_acceptable` |
| 4. Validate / re-segment | `SpeechAlignmentPipeline.validate_segments` | `list[SegmentValidation]` |

**Under the hood** (you never call these directly): `WhisperWrapper` (ASR) →
`TorchaudioForcedAligner` or `WhisperxForcedAligner` (word times) →
`find_segment` (rapidfuzz) for re-alignment; metrics use the standard Whisper
normalizer + `xml-roberta` for semantic similarity.

**Dataclasses returned**

| Type | Fields |
|---|---|
| `Word` | `text, start, end, score` |
| `TranscriptMetrics` | `wer, cer, similarity, normalized_wer, normalized_cer, normalized_similarity, semantic_similarity, reference, hypothesis` |
| `SegmentMatch` | `text, start, end, score, word_start_idx, word_end_idx` |
| `SegmentValidation` | `text, annotated_start/end, recovered_start/end, start/end_offset, score, decision, match` |

Demo audio: `fi_protagonist.wav` (a YouTube-style monologue).

In [1]:
from pathlib import Path

from exordium.audio.io import load_audio

audio_path = Path("../tests/fixtures/audio/fi_protagonist.wav")
waveform, sr = load_audio(audio_path, target_sample_rate=16000)
print(f"Audio file : {audio_path}  (exists={audio_path.exists()})")
print(f"Duration   : {waveform.shape[-1] / sr:.1f} s @ {sr} Hz mono")

Audio file : ../tests/fixtures/audio/fi_protagonist.wav  (exists=True)
Duration   : 15.3 s @ 16000 Hz mono


## 0. Build the pipeline (backend hidden behind one object)

In [2]:
from exordium.text.pipeline import SpeechAlignmentPipeline

# backend defaults to "whisperx" (wav2vec2 alignment); "torchaudio" (MMS_FA) also
# available. device_id: -1/None -> CPU, 0+ -> GPU/MPS.
pipe = SpeechAlignmentPipeline(language="en", device_id=-1)
print("pipeline ready — backend:", pipe.backend)

/Volumes/UGREEN/Dev/exordium/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/539 [00:00<?, ?it/s]

Loading weights:   0%|          | 2/539 [00:00<01:00,  8.93it/s]

Loading weights:  34%|███▍      | 183/539 [00:00<00:00, 710.43it/s]

Loading weights:  54%|█████▍    | 290/539 [00:00<00:00, 720.50it/s]

Loading weights:  71%|███████   | 382/539 [00:00<00:00, 748.70it/s]

Loading weights:  87%|████████▋ | 470/539 [00:00<00:00, 731.06it/s]

Loading weights: 100%|██████████| 539/539 [00:00<00:00, 672.36it/s]

pipeline ready — backend: whisperx


## 1. Audio → transcript

`transcribe` returns a plain string (this is `WhisperWrapper` underneath).

In [3]:
transcript = pipe.transcribe(audio_path)
print(f"{type(transcript).__name__}, {len(transcript)} chars (full text below):\n\n{transcript}")

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


str, 226 chars (full text below):

video is going to be a quick one but it's going to be a great one for you guys so let's get started so yes as you can see from the title I hit 10,000 subscribers and you guys just don't know how much that means to me I am very


## 2. Audio → word-level timestamps

`transcribe_words` runs ASR then forced alignment and returns `list[Word]`,
each carrying `start` / `end` (seconds) and an alignment `score`.

In [4]:
words = pipe.transcribe_words(audio_path)
print(f"{len(words)} words. First 12 (each is a Word dataclass):\n")
print(f"{'word':<14}{'start':>8}{'end':>8}{'score':>8}")
for w in words[:12]:
    print(f"{w.text:<14}{w.start:>8.2f}{w.end:>8.2f}{w.score:>8.2f}")

51 words. First 12 (each is a Word dataclass):

word             start     end   score
video             0.00    0.71    0.69
is                0.79    0.87    0.65
going             0.95    1.15    0.86
to                1.19    1.27    0.59
be                1.33    1.49    0.87
a                 1.51    1.53    1.00
quick             1.85    2.14    0.97
one               2.38    2.48    0.75
but               2.84    2.96    0.83
it's              3.02    3.10    0.28
going             3.12    3.25    0.34
to                3.29    3.33    0.91


## 3. Check an annotation against the prediction

`TranscriptEvaluator` compares two strings — a reference (annotation) and a
hypothesis (prediction) — and returns `TranscriptMetrics`. It is **independent of
audio, ASR, and the pipeline** (loads only a sentence-embedding model), so you can
call it on any two strings. *(The pipeline also offers
`pipe.evaluate_annotation(audio, text)`, which transcribes then calls this.)*

Metric views (WER/CER: lower = better, 0 = identical; sim/semantic: higher = better):

* **raw** — lightly normalized; keeps apostrophes/digits, so `"ten thousand"` vs
  `"10,000"` count as errors. Inflated by formatting.
* **normalized** — via **Whisper's standard `EnglishTextNormalizer`** (from
  `transformers`; the Whisper paper / HF Open ASR Leaderboard routine — not
  hand-rolled). Canonicalizes numbers/contractions/casing. **Judge on this.**
* **sim** (0–100) — `rapidfuzz` character-level string similarity (lexical, no model).
* **semantic** (−1…1) — cosine of **xml-roberta** sentence embeddings (meaning-level,
  robust to paraphrase).

First a **sanity check**: feeding the *same* string as both sides must be a perfect
match (0 WER, semantic 1.0).

In [5]:
from exordium.text.evaluation import TranscriptEvaluator, is_acceptable

evaluator = TranscriptEvaluator()  # xml-roberta semantic + standard normalizer

# SANITY CHECK: identical strings MUST be a perfect match.
sanity = evaluator.evaluate(transcript, transcript)
print("sanity — transcript vs itself (identical):")
print(
    f"  normalized : wer={sanity.normalized_wer:.3f}  cer={sanity.normalized_cer:.3f}  "
    f"sim={sanity.normalized_similarity:.1f}"
)
print(f"  semantic   : cosine={sanity.semantic_similarity:.3f}")
assert sanity.normalized_wer == 0.0 and sanity.semantic_similarity > 0.999
print("  -> perfect match (0 WER, semantic 1.0), as expected ✓")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8533.11it/s]

sanity — transcript vs itself (identical):
  normalized : wer=0.000  cer=0.000  sim=100.0
  semantic   : cosine=1.000
  -> perfect match (0 WER, semantic 1.0), as expected ✓


### Accept rule & defaults

`is_acceptable(metrics)` gives a heuristic verdict — accept if **either**:

* `normalized_wer ≤ 0.20` (lexically close — the standard word-level criterion), **or**
* `semantic_similarity ≥ 0.70` (clearly related/paraphrase — rescues high-WER but
  same-meaning pairs).

Both are keyword-overridable defaults (`DEFAULT_MAX_NORMALIZED_WER`,
`DEFAULT_MIN_SEMANTIC_SIMILARITY` in `exordium.text.evaluation`) — there is no
universal ASR cutoff, so tune per dataset (e.g. raise WER to ~0.35 for "fair").
CER and the raw metrics are reported for insight but not part of the decision.

Now the **real** comparison — your annotation vs the prediction:

In [6]:
# The transcript we believe is in the clip (the "annotation") — written casually
# with a spelled-out number, as annotations often are.
annotated = (
    "todays video is going to be a quick one, but it gonna be a great one for you "
    "guys, so lets get started. So yes, as you can see from the title, I hit ten "
    "thousand subscribers and you guys just dont know how much that means to me. Im very"
)

m = evaluator.evaluate(annotated, transcript)  # reference (annotation) vs hypothesis (prediction)
print("real — annotation vs prediction:")
print(f"  raw        : wer={m.wer:.3f}  cer={m.cer:.3f}  sim={m.similarity:.1f}")
print(
    f"  normalized : wer={m.normalized_wer:.3f}  cer={m.normalized_cer:.3f}  "
    f"sim={m.normalized_similarity:.1f}"
)
print(f"  semantic   : cosine={m.semantic_similarity:.3f}  (meaning-level)")
print(f"  is_acceptable -> {is_acceptable(m)}")

real — annotation vs prediction:
  raw        : wer=0.196  cer=0.126  sim=92.1
  normalized : wer=0.157  cer=0.071  sim=96.5
  semantic   : cosine=0.902  (meaning-level)
  is_acceptable -> True


## 4. Validate annotated segments & recover cut points

You have utterance-level annotations `(text, start, end)` and want to check each
one sits where the annotation claims — and, if not, get the corrected times.
`validate_segments` runs **one** ASR pass, then fuzzy-locates every segment.

Each result is a `SegmentValidation` with a `decision`:

* `accept` — annotation within `tolerance` of the true span,
* `recut`  — text found but shifted → use `recovered_start` / `recovered_end`,
* `drop`   — text not found (likely a wrong edit).

In [7]:
# We locate a real phrase to build a KNOWN-GOOD annotation (guaranteed accept),
# then add a deliberately shifted one (recut) and an absent one (drop).
from exordium.text.transcript_align import find_segment

phrase = "so lets get started"
good = find_segment(phrase, words)
print(f"located {phrase!r} at {good.start:.2f}-{good.end:.2f}s (score {good.score:.0f})\n")

segments = [
    # (text, annotated_start, annotated_end)
    (phrase, good.start, good.end),  # -> accept (annotation matches)
    (phrase, good.start + 20.0, good.end + 20.0),  # -> recut  (annotation shifted +20s)
    ("please smash that subscribe button right now", 1.0, 4.0),  # -> drop (not spoken)
]

results = pipe.validate_segments(audio_path, segments, tolerance=0.3)

for v in results:
    print(
        f"decision = {v.decision.upper():7}  score={v.score:5.1f}  "
        f"annotated=({v.annotated_start:.2f},{v.annotated_end:.2f})  "
        f"recovered={None if v.recovered_start is None else f'({v.recovered_start:.2f},{v.recovered_end:.2f})'}"
    )
    if v.decision == "recut":
        print(
            f"           -> cut here: start={v.recovered_start:.2f}s end={v.recovered_end:.2f}s "
            f"(offset {v.start_offset:+.2f}s)"
        )
    print(f"           text: {v.text!r}")

located 'so lets get started' at 5.46-6.73s (score 95)



decision = ACCEPT   score= 94.7  annotated=(5.46,6.73)  recovered=(5.46,6.73)
           text: 'so lets get started'
decision = RECUT    score= 94.7  annotated=(25.46,26.73)  recovered=(5.46,6.73)
           -> cut here: start=5.46s end=6.73s (offset -20.00s)
           text: 'so lets get started'
decision = DROP     score=  0.0  annotated=(1.00,4.00)  recovered=None
           text: 'please smash that subscribe button right now'


### Reading the structured result

`recovered_start` / `recovered_end` are the corrected cut points — hand them to
your own audio-cutting step whenever `decision != "drop"`. Everything is a plain
dataclass, so it serializes cleanly (`dataclasses.asdict`) for logging or JSON.

In [8]:
import json
from dataclasses import asdict

# 'match' holds the underlying SegmentMatch; drop it for a clean JSON-able view.
clean = {k: v for k, v in asdict(results[1]).items() if k != "match"}
print(json.dumps(clean, indent=2))

{
  "text": "so lets get started",
  "annotated_start": 25.463,
  "annotated_end": 26.732,
  "recovered_start": 5.463,
  "recovered_end": 6.732,
  "start_offset": -20.0,
  "end_offset": -20.0,
  "score": 94.73684210526316,
  "decision": "recut"
}


## 5. Swap the alignment backend — same API, torchaudio word times

The default backend above is **whisperX**. Switching to `backend="torchaudio"`
(``MMS_FA``) uses a different forced aligner but returns an identical
`list[Word]`, so every method works unchanged — handy for cross-checking timings.

We pass `whisper=pipe.whisper` so the second pipeline **reuses** the already
-loaded Whisper model instead of loading a second ~1.5 GB copy (Whisper is by
far the largest model here — this keeps memory in check).

In [9]:
pipe_ta = SpeechAlignmentPipeline(
    backend="torchaudio", language="en", device_id=-1, whisper=pipe.whisper
)
words_ta = pipe_ta.transcribe_words(audio_path)

print(f"{'word':<14}{'whisperX':>22}{'torchaudio':>22}")
for a, b in list(zip(words, words_ta))[:8]:
    print(f"{a.text:<14}{f'{a.start:.2f}-{a.end:.2f}':>22}{f'{b.start:.2f}-{b.end:.2f}':>22}")

word                        whisperX            torchaudio
video                      0.00-0.71             0.18-0.52
is                         0.79-0.87             0.74-0.84
going                      0.95-1.15             0.90-1.12
to                         1.19-1.27             1.16-1.24
be                         1.33-1.49             1.29-1.37
a                          1.51-1.53             1.57-1.59
quick                      1.85-2.14             1.75-2.07
one                        2.38-2.48             2.27-2.51


## 6. Correctness checks — the assertions CI no longer runs

Everything above *demonstrates* the pipeline; this section *asserts* it. These checks used
to live in `tests/text/`, but they need the **real weights** — with random weights they
would assert nothing meaningful. Correctness is a property of the weights, not the code,
so it is checked here instead of in CI (which builds every wrapper with `pretrained=False`
and downloads no checkpoints).

Two of these guard bugs that **shipped once already**: Whisper cropping long audio at its
30 s window, and forced alignment crashing on degenerate micro-segments. Run this notebook
after touching `WhisperWrapper`, `TorchaudioForcedAligner`, or `SpeechAlignmentPipeline` —
the cells fail loudly if either returns.

In [10]:
# The long-form checks need audio LONGER than Whisper's 30 s window. The notebook above
# uses a 15 s clip, so load the 61 s multispeaker fixture explicitly here.
from exordium.text.transcript_align import find_segment

long_path = Path("../tests/fixtures/audio/multispeaker.wav")
long_wave, long_sr = load_audio(long_path, target_sample_rate=16000)
long_duration = long_wave.shape[-1] / long_sr
assert long_duration > 30.0, "fixture must exceed the 30 s window or this proves nothing"

long_words = pipe.transcribe_words(long_path)
print(
    f"{long_duration:.1f} s audio -> {len(long_words)} words, "
    f"last ends at {long_words[-1].end:.1f} s"
)

assert len(long_words) > 180, f"long-form decode regressed: only {len(long_words)} words"
assert long_words[-1].end > 45.0, "word stream stops early — audio past 30 s was lost"
starts = [w.start for w in long_words]
assert starts == sorted(starts), "words are not chronological"

# A phrase from the *end* of the recording must be findable — i.e. the tail survived.
late = find_segment("check us out at my taste base", long_words)
assert late is not None, "text near the end of the audio is missing from the word stream"
assert late.start > 45.0, f"late phrase found at {late.start:.1f} s — too early to be the tail"
print(f"late phrase located at {late.start:.1f}-{late.end:.1f} s  (score {late.score:.0f})")

# And a mid-recording phrase must fuzzy-match with high coverage.
mid = find_segment("what do you guys want to eat for lunch", long_words)
assert mid is not None and mid.score >= 80.0, "known phrase did not match the word stream"
print(f"mid phrase   located at {mid.start:.1f}-{mid.end:.1f} s  (score {mid.score:.0f})")

61.2 s audio -> 254 words, last ends at 60.6 s
late phrase located at 57.8-60.4 s  (score 86)
mid phrase   located at 0.9-2.0 s  (score 100)


In [11]:
# The semantic embedder must actually rank meaning, not just return numbers.
identical = evaluator.evaluate("hello world", "hello world")
assert identical.semantic_similarity is not None
assert identical.semantic_similarity > 0.95, "identical text should be ~1.0 similar"

para = evaluator.evaluate("the cat sat on the mat", "a cat was sitting on a mat")
unrel = evaluator.evaluate("the cat sat on the mat", "quarterly revenue exceeded forecasts")
print(f"identical  {identical.semantic_similarity:.3f}")
print(f"paraphrase {para.semantic_similarity:.3f}")
print(f"unrelated  {unrel.semantic_similarity:.3f}")
assert para.semantic_similarity > unrel.semantic_similarity, "paraphrase must beat unrelated"

print("\nAll alignment correctness checks passed.")

identical  1.000
paraphrase 0.980
unrelated  -0.001

All alignment correctness checks passed.


In [12]:
# Forced alignment must survive degenerate micro-segments (real text, ~0.02 s span), and
# must recover the word *correctly timed* — not merely avoid the crash. These need the real
# weights: with random weights the recovered timing is meaningless, so CI cannot check it.
from exordium.text.alignment import TorchaudioForcedAligner
from exordium.text.base import Segment

aligner = TorchaudioForcedAligner(device_id=-1)

# 1) A degenerate segment must not crash the recording, and must keep its word.
segments = [
    Segment(text="hey guys", start=0.0, end=2.0),
    Segment(text="that", start=2.100, end=2.120),  # 320 samples — degenerate
    Segment(text="I", start=2.200, end=2.230),  # 480 samples — degenerate
    Segment(text="what do you guys want to eat", start=2.5, end=6.0),
]
aligned = aligner.align_segments(long_path, segments)
text = [w.text.strip().lower() for w in aligned]
print(f"{len(aligned)} words from {len(segments)} segments (2 degenerate)")
assert "that" in text and "i" in text, "a degenerate micro-segment lost its word"
assert [w.start for w in aligned] == sorted(w.start for w in aligned), "words not chronological"

# 2) The recovered word must land where it really is. Align the full sentence for ground
#    truth, then re-align one of its words as a degenerate 0.02 s micro-segment.
sentence = "Hey guys, what do you guys want to eat for lunch?"
truth = aligner.align(long_path, sentence)
anchored = [w for w in truth if 0.05 <= (w.end - w.start) <= 0.4]
target = max(anchored, key=lambda w: w.score)  # a tightly-bounded, confident word

midpoint = (target.start + target.end) / 2
recovered = aligner.align_segments(
    long_path, [Segment(text=target.text, start=midpoint, end=midpoint + 0.02)]
)
assert len(recovered) == 1
delta = abs(recovered[0].start - target.start)
print(
    f"{target.text!r}: truth {target.start:.3f}s -> recovered {recovered[0].start:.3f}s "
    f"(off by {delta * 1000:.0f} ms)"
)
assert delta < 0.1, f"recovered word drifted {delta:.3f}s from where it really is"

print("\nAll alignment correctness checks passed.")

11 words from 4 segments (2 degenerate)


'to': truth 51.167s -> recovered 51.191s (off by 24 ms)

All alignment correctness checks passed.
